# 

In [1]:
import numpy as np
import adi
import matplotlib.pyplot as plt
import subprocess
from time import sleep
                

    
    

default = {"i_ampl":           (0x43C00000,   0x0000FFFF),
            "q_ampl":           (0x43C00004,  0x0000FFFF),
            "frequency":        (0x43C0000C,  59910463),
            "multiplier":       (0x43C00014,  42),
            "Phase_PDH":        (0x43C00010,  0x00100000),
            "phase_difference": (0x43C00008,  0x00100000)}





settings_3MHz = {"i_ampl":           (0x43C00000,  0x0000FFFF),
            "q_ampl":           (0x43C00004,  0xe7c7 ), #0x0000ED3F 0xdff7
            "frequency":        (0x43C0000C,  27306666),
            "multiplier":       (0x43C00014,  42),
            "Phase_PDH":        (0x43C00010,  0x00100000),
            "phase_difference": (0x43C00008,  1028576  )} #

settings_20MHz = {"i_ampl":           (0x43C00000,  0x0000FFFF),
            "q_ampl":           (0x43C00004,  0x0000FFFF ), #0x0000ED3F 0xdff7
            "frequency":        (0x43C0000C,  174762666),
            "multiplier":       (0x43C00014,  42),
            "Phase_PDH":        (0x43C00010,  0X00100000),
            "phase_difference": (0x43C00008,  1071576  )} #1084976

settings = default
 
    
    
sample_rate = 61.44e6 # Hz

freq_min = 750_000_000
freq_max = 2_250_000_000

q_ampl_correction_low_3MHz = 0x0000_E7C7
q_ampl_correction_high_3MHz = 0x0000_ED3F

phase_correction_low_3MHz = 1038576
phase_correction_high_3MHz = 1028576

q_ampl_correction_low_20MHz = 0x0000_FFFF
q_ampl_correction_high_20MHz = 0x0000_FFFF

phase_correction_low_20MHz = 1084976
phase_correction_high_20MHz = 1071576

q_ampl_correction_low_7MHz = 0x0000_f1bd
q_ampl_correction_high_7MHz = 0x0000_FFFF

phase_correction_low_7MHz = 1040576
phase_correction_high_7MHz = 995376


def external_clock():
    subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo 954  > /sys/class/gpio/export'",shell=True)
    subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo out >  /sys/class/gpio/gpio954/direction'",shell=True)
    subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo 1 >  /sys/class/gpio/gpio954/value'",shell=True)
    
def internal_clock():
    subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo 954  > /sys/class/gpio/export'",shell=True)
    subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo out >  /sys/class/gpio/gpio954/direction'",shell=True)
    subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo 0 >  /sys/class/gpio/gpio954/value'",shell=True)



def interpolate(frequency, correction_low = 0x0000_E7C7, correction_high = 0x0000_ED3F):
    return int(correction_low + (frequency - freq_min) / (freq_max - freq_min) * (correction_high - correction_low) )


def change_frequency(frequency,q_ampl_correction_low = q_ampl_correction_low_3MHz ,
                     q_ampl_correction_high = q_ampl_correction_high_3MHz ,
                     phase_correction_low = phase_correction_low_3MHz ,
                     phase_correction_high = phase_correction_high_3MHz ):
    amplitude_q =  interpolate(frequency, q_ampl_correction_low, q_ampl_correction_high)
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['q_ampl'][0]} 32 {amplitude_q}'",shell=True)

    phase_difference = interpolate(frequency, phase_correction_low, phase_correction_high)             
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['phase_difference'][0]} 32 {phase_difference}'",shell=True)

    sdr.tx_lo = frequency
    
def change_mod_frequency(frequency, sampling_rate = 28_000_000):    
    frequency_code = int(frequency * 0xFFFF_FFFF / sampling_rate / 8) # divided by 8 because the PDH generator runs 8 times faster
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['frequency'][0]} 32 {frequency_code}'",shell=True)
    print(frequency_code)
    
def change_mod_depth(depth=42):
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['multiplier'][0]} 32 {depth}'",shell=True)
    
    
# Configuring the FPGA    
subprocess.call("sshpass -p 'analog' scp system_top.bit.bin root@192.168.2.1:/lib/firmware/.",shell=True)
subprocess.call("sshpass -p 'analog' scp configure_FPGA.sh  root@192.168.2.1:/root",shell=True)               
subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'sh configure_FPGA.sh'",shell=True)


sdr = adi.Pluto("ip:192.168.2.1")
sdr.sample_rate = int(sample_rate)





Matplotlib created a temporary config/cache directory at /tmp/matplotlib-yh1towr4 because the default path (/home/fpga/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
settings = default

for key,value in settings.items():
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {value[0]} 32 {value[1]}'",shell=True)


# Config Tx
sdr.tx_rf_bandwidth = int(sample_rate) # filter cutoff, just set it to the same as sample rate

sdr.tx_hardwaregain_chan0 = 0 # Increase to increase tx power, valid range is -90 to 0 dB
# change_frequency(2_500_000_000)
change_frequency(750_000_000)
subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['phase_difference'][0]} 32 {1090576}'",shell=True)

# Start the transmitter
i = np.zeros(16) #Generating "fake" samples to enable transmitter
q = np.zeros(16)
samples= i+1j*q
sdr.tx_cyclic_buffer = True # Enable cyclic buffers
sdr.tx(samples) # start transmitting



In [3]:
sdr.sample_rate = int(61.44e6)
sdr.tx_rf_bandwidth = int(20e6)

change_mod_depth(36)

In [4]:
change_frequency(1000_000_000,q_ampl_correction_low_3MHz,
                              q_ampl_correction_high_3MHz,
                              phase_correction_low_3MHz,
                              phase_correction_high_3MHz)

In [ ]:
settings_7MHz = {"i_ampl":           (0x43C00000,   0x0000FFFF),
            "q_ampl":           (0x43C00004,  0xffff),#0xf1bd
            "frequency":        (0x43C0000C,  68266666),
            "multiplier":       (0x43C00014,  30),
            "Phase_PDH":        (0x43C00010,  0x00100000),
            "phase_difference": (0x43C00008,  995376)}#1040576
for key,value in settings_20MHz.items():
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {value[0]} 32 {value[1]}'",shell=True)

In [ ]:
change_mod_frequency(3.84000001e6,sampling_rate = 61.44e6) #7.8125e6

In [ ]:
def manual_phase_calibration(interval=100,gain=200000,center = 0x00100000):
    for value in range(interval):
        phase_diff = int(center+(value/interval-0.5)*gain)
        subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['phase_difference'][0]} 32 {phase_diff}'",shell=True)
        print(phase_diff)
        sleep(0.2)

def manual_ampl_calibration(interval = 10000,step = 100):   
    for value in range(0,interval,step):
        ampl = 0x0000FFFF - interval + value + step
        subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['i_ampl'][0]} 32 {ampl}'",shell=True)
        print(hex(ampl))
        sleep(0.2)
        
# manual_ampl_calibration(interval = 10000,step = 50)
# manual_phase_calibration(interval=50,gain=20000,center=995376)


In [7]:
jump = 2000
samples = 100
center_freq = 1e9
while True:
    for n in range(samples):
        sdr.tx_lo = int(center_freq-jump*samples//2+jump*2*n)
        sleep(0.5)


ERROR: READ LINE: -32


BrokenPipeError: [Errno 32] Broken pipe

In [ ]:
# Stop transmitting
sdr.tx_destroy_buffer()

In [ ]:
print(int(0x0000FFFF))

In [ ]:
sdr.tx_rf_bandwidth = int(sample_rate)

In [ ]:
phase_noise_settings = {"i_ampl":           (0x43C00000,   0x0000FFFF),
            "q_ampl":           (0x43C00004,  0x0000FFFF),
            "frequency":        (0x43C0000C,  59910463),
            "multiplier":       (0x43C00014,  0),
            "Phase_PDH":        (0x43C00010,  0x00100000),
            "phase_difference": (0x43C00008,  0x00100000)}
for key,value in phase_noise_settings.items():
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {value[0]} 32 {value[1]}'",shell=True)
    
change_frequency(2250_000_000)
    

In [ ]:
import iio
ctx = iio.Context("ip:192.168.2.1")

In [ ]:
phy = sdr.ctx.find_device("ad9361-phy")
for dattr in phy.debug_attrs:
    print(dattr, phy.debug_attrs[dattr])

In [ ]:
phy = sdr.ctx.find_device("ad9361-phy")
phy.debug_attrs["adi,xo-disable-use-ext-refclk-enable"].value = '1'


In [ ]:
internal_clock()

In [ ]:
external_clock()

In [ ]:
def change_frequency(frequency,q_ampl_correction_low, q_ampl_correction_high,phase_correction_low, phase_correction_high):
    amplitude_q =  interpolate(frequency, q_ampl_correction_low, q_ampl_correction_high)
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['q_ampl'][0]} 32 {amplitude_q}'",shell=True)

    phase_difference = interpolate(frequency, phase_correction_low, phase_correction_high)             
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {settings['phase_difference'][0]} 32 {phase_difference}'",shell=True)

    sdr.tx_lo = frequency

In [ ]:
sdr.tx_hardwaregain_chan0 = 0 # Increase to increase tx power, valid range is -90 to 0 dB

In [ ]:
amplitudes = { "i_ampl":           (0x43C00000,  0x0000FFFF//2),
              "q_ampl":           (0x43C00004,  0x0000FFFF//2 ) }
for key,value in amplitudes.items():
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {value[0]} 32 {value[1]}'",shell=True)

In [ ]:
change_mod_depth(100)

In [ ]:
manual_ampl_calibration()